In [1]:
# ==========================================
# IMPORTAR LIBRERÍAS
# ==========================================

# Librería para análisis de datos
import pandas as pd

# Librería para operaciones numéricas
import numpy as np

In [2]:
# ==========================================
# CARGAR DATASET
# ==========================================

# URL RAW del dataset en GitHub
url = "https://raw.githubusercontent.com/MrAndels/energyiq/refs/heads/main/data/raw/energy_dataset_reducido.csv"

# Leer dataset CSV
df = pd.read_csv(
    url,
    
    # Separador de columnas
    sep=";",
    
    # Evita problemas de memoria
    low_memory=False
)

# Mostrar primeras filas
df.head()

,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,16/12/2006,17:24:00,4.216,0.418,234.840,18.400,0.000,1.000,17.0
1,16/12/2006,17:25:00,5.360,0.436,233.630,23.000,0.000,1.000,16.0
2,16/12/2006,17:26:00,5.374,0.498,233.290,23.000,0.000,2.000,17.0
3,16/12/2006,17:27:00,5.388,0.502,233.740,23.000,0.000,1.000,17.0
4,16/12/2006,17:28:00,3.666,0.528,235.680,15.800,0.000,1.000,17.0


In [3]:
# ==========================================
# CONVERSIÓN DE TIPOS DE DATOS
# ==========================================

# Crear columna completa de fecha y hora
df["Datetime"] = pd.to_datetime(
    
    # Unir columnas Date y Time
    df["Date"] + " " + df["Time"],
    
    # Formato europeo de fecha
    dayfirst=True
)

# Lista de columnas numéricas
columnas_numericas = [
    "Global_active_power",
    "Global_reactive_power",
    "Voltage",
    "Global_intensity",
    "Sub_metering_1",
    "Sub_metering_2",
    "Sub_metering_3"
]

# Convertir columnas a números
for columna in columnas_numericas:
    
    # Convertir texto a tipo numérico
    df[columna] = pd.to_numeric(
        df[columna],
        
        # Convertir errores en NaN
        errors="coerce"
    )

# Mostrar tipos actualizados
print(df.dtypes)

Date                                str
Time                                str
Global_active_power             float64
Global_reactive_power           float64
Voltage                         float64
Global_intensity                float64
Sub_metering_1                  float64
Sub_metering_2                  float64
Sub_metering_3                  float64
Datetime                 datetime64[us]
dtype: object


In [4]:
# ==========================================
# MANEJO DE VALORES NULOS
# ==========================================

# Mostrar cantidad de valores nulos
print("Valores nulos por columna:")
print(df.isnull().sum())

# Mostrar porcentaje de valores nulos
print("\nPorcentaje de valores nulos:")

# Calcular porcentaje
print((df.isnull().sum() / len(df)) * 100)

Valores nulos por columna:
Date                        0
Time                        0
Global_active_power      8350
Global_reactive_power    8350
Voltage                  8350
Global_intensity         8350
Sub_metering_1           8350
Sub_metering_2           8350
Sub_metering_3           8350
Datetime                    0
dtype: int64

Porcentaje de valores nulos:
Date                     0.000000
Time                     0.000000
Global_active_power      0.521875
Global_reactive_power    0.521875
Voltage                  0.521875
Global_intensity         0.521875
Sub_metering_1           0.521875
Sub_metering_2           0.521875
Sub_metering_3           0.521875
Datetime                 0.000000
dtype: float64


In [5]:
# ==========================================
# ELIMINAR VALORES NULOS
# ==========================================

# Mostrar dimensiones antes de limpiar
print("Dimensiones antes:")
print(df.shape)

# Eliminar filas con valores nulos
df = df.dropna()

# Mostrar dimensiones después de limpiar
print("\nDimensiones después:")
print(df.shape)

# Verificar valores nulos restantes
print("\nValores nulos restantes:")
print(df.isnull().sum())

Dimensiones antes:
(1600000, 10)

Dimensiones después:
(1591650, 10)

Valores nulos restantes:
Date                     0
Time                     0
Global_active_power      0
Global_reactive_power    0
Voltage                  0
Global_intensity         0
Sub_metering_1           0
Sub_metering_2           0
Sub_metering_3           0
Datetime                 0
dtype: int64


In [6]:
# ==========================================
# ELIMINAR FILAS DUPLICADAS
# ==========================================

# Mostrar dimensiones antes
print("Dimensiones antes:")
print(df.shape)

# Eliminar filas duplicadas
df = df.drop_duplicates()

# Mostrar dimensiones después
print("\nDimensiones después:")
print(df.shape)

# Verificar duplicados restantes
print("\nDuplicados restantes:")
print(df.duplicated().sum())

Dimensiones antes:
(1591650, 10)

Dimensiones después:
(1591650, 10)

Duplicados restantes:
0


In [7]:
# ==========================================
# OUTLIERS EN TODAS LAS VARIABLES
# ==========================================

# Lista de columnas numéricas
columnas_numericas = [
    "Global_active_power",
    "Global_reactive_power",
    "Voltage",
    "Global_intensity",
    "Sub_metering_1",
    "Sub_metering_2",
    "Sub_metering_3"
]

# Analizar outliers por columna
for columna in columnas_numericas:
    
    # Calcular cuartil 1
    Q1 = df[columna].quantile(0.25)

    # Calcular cuartil 3
    Q3 = df[columna].quantile(0.75)

    # Calcular rango intercuartil
    IQR = Q3 - Q1

    # Calcular límites
    limite_inferior = Q1 - 1.5 * IQR
    limite_superior = Q3 + 1.5 * IQR

    # Detectar outliers
    outliers = df[
        (df[columna] < limite_inferior) |
        (df[columna] > limite_superior)
    ]

    # Mostrar resultados
    print(f"\nVariable: {columna}")
    print(f"Cantidad de outliers: {outliers.shape[0]}")


Variable: Global_active_power
Cantidad de outliers: 79159

Variable: Global_reactive_power
Cantidad de outliers: 30686

Variable: Voltage
Cantidad de outliers: 34792

Variable: Global_intensity
Cantidad de outliers: 85940

Variable: Sub_metering_1
Cantidad de outliers: 132568

Variable: Sub_metering_2
Cantidad de outliers: 64751

Variable: Sub_metering_3
Cantidad de outliers: 0


In [8]:
# ==========================================
# CREACIÓN DE VARIABLES TEMPORALES
# ==========================================

# Extraer año
df["Year"] = df["Datetime"].dt.year

# Extraer mes
df["Month"] = df["Datetime"].dt.month

# Extraer día
df["Day"] = df["Datetime"].dt.day

# Extraer hora
df["Hour"] = df["Datetime"].dt.hour

# Extraer día de la semana
df["Weekday"] = df["Datetime"].dt.day_name()

# Mostrar primeras filas
print(df.head())

         Date      Time  Global_active_power  Global_reactive_power  Voltage  \
0  16/12/2006  17:24:00                4.216                  0.418   234.84   
1  16/12/2006  17:25:00                5.360                  0.436   233.63   
2  16/12/2006  17:26:00                5.374                  0.498   233.29   
3  16/12/2006  17:27:00                5.388                  0.502   233.74   
4  16/12/2006  17:28:00                3.666                  0.528   235.68   

   Global_intensity  Sub_metering_1  Sub_metering_2  Sub_metering_3  \
0              18.4             0.0             1.0            17.0   
1              23.0             0.0             1.0            16.0   
2              23.0             0.0             2.0            17.0   
3              23.0             0.0             1.0            17.0   
4              15.8             0.0             1.0            17.0   

             Datetime  Year  Month  Day  Hour   Weekday  
0 2006-12-16 17:24:00  2006     12

In [9]:
# ==========================================
# GUARDAR DATASET LIMPIO
# ==========================================

# Guardar dataset limpio en CSV
df.to_csv(
    
    # Ruta del archivo
    "../data/processed/energy_dataset_clean.csv",
    
    # No guardar índice
    index=False
)

# Mensaje de confirmación
print("Dataset limpio guardado correctamente")

Dataset limpio guardado correctamente
